In [6]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

c:\Users\PC\Desktop\python\rates_degree_work


- La tabla es semanal, terminando en cada viernes

In [7]:
tasascap = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascap.columns = [x.lower().replace(" ","_") for x in tasascap.columns]
tasascap.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_obl", 
                           "sum_numero_de_creditos":"num_obl", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascap.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascap.sample(5)

,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa,margen_adicional,monto,num_obl,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio
67789,BC-ESTABLECIMIENTO BANCARIO,AV Villas,28/10/2022,Natural,Femenino,No aplica,Consumo,Garantia idónea o no idónea,Tarjeta de crédito para ingresos hasta 2 SMMLV,Avances en efectivo,36.8700,0.0000,7140000.0000,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85026,CF-COMPAÑÍA DE FINANCIAMIENTO,Coltefinanciera,25/11/2022,Natural,Femenino,No aplica,Consumo,Sin garantia,Libranza otros,Más de 5 años y hasta 7 años,30.8700,0.0000,36369238.0000,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34686,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,26/08/2022,Jurídica,No aplica,Pequeña empresa,Comercial ordinario,Garantia idónea o no idónea,Construcción de proyectos de vivienda no vis (colocación en uvr),Más de 30 días y hasta 1 año,16.1600,5.4000,3540705373.3200,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56290,BC-ESTABLECIMIENTO BANCARIO,Banco de Bogotá,07/10/2022,Natural,Femenino,No aplica,Consumo,Sin garantia,Consumo empleados,Más de 5 años y hasta 7 años,12.7500,0.0000,39700000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60192,COOPERATIVAS DE CARÁCTER FINANCIERO,Coofinep,14/10/2022,Natural,Femenino,No aplica,Consumo,Sin garantia,Crédito rotativo,Entre 31 días y 1 año,31.3700,0.0000,100000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio']
cols_cont = ['tasa', 'margen_adicional', 'monto', 'num_obl', 'rango_monto_desembolsado']

tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"], dayfirst=True, errors='coerce')

tasascap["tasa"] = tasascap["tasa"].astype(float)
tasascap["num_obl"] = tasascap["num_obl"].astype(int)
tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"])
tasascap["anomes"] = tasascap["fecha_corte"].dt.strftime("%Y-%m")

# tasascap = tasascap[~tasascap["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascap = tasascap[tasascap["tasa"] <= 50]
# tasascap = tasascap[tasascap["num_obl"] >= 2]
# tasascap = tasascap[(tasascap["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascap ["banco"] = tasascap["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascap["plazo"] = tasascap["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascap["prom_monto_porobl"] = tasascap["monto"]/tasascap["num_obl"]

print(tasascap["fecha_corte"].min(), "|", tasascap["fecha_corte"].max())

print(tasascap.shape)
print("Bancos:", tasascap["banco"].nunique())

tasascap.sample(7)

2022-07-01 00:00:00 | 2022-12-30 00:00:00
(107151, 23)
Bancos: 49


,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa,margen_adicional,monto,num_obl,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio,anomes,prom_monto_porobl
90404,BC-ESTABLECIMIENTO BANCARIO,Bancoomeva,2022-12-02,Jurídica,No aplica,Pequeña empresa,Comercial ordinario,Garantia idónea o no idónea,Sobregiro en cuenta corriente persona jurídica o persona natural,(0-1],24.7700,0.0000,99943227.0000,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-12,14277603.8571
17006,CF-COMPAÑÍA DE FINANCIAMIENTO,Coltefinanciera,2022-07-29,Natural,Femenino,Microempresa,Comercial ordinario,Sin garantia,Otros Comercial,(5-7],29.5700,0.0000,84870000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07,84870000.0000
53278,BC-ESTABLECIMIENTO BANCARIO,Banco Pichincha S.A.,2022-09-30,Natural,Femenino,No aplica,Consumo,Sin garantia,Libranza otros,(1-3],20.4100,0.0000,5000000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-09,5000000.0000
75719,BC-ESTABLECIMIENTO BANCARIO,Coopcentral,2022-11-11,Natural,Masculino,No aplica,Consumo,Sin garantia,Libranza otros,(3-5],16.5000,0.0000,35000000.0000,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-11,11666666.6667
103106,CF-COMPAÑÍA DE FINANCIAMIENTO,Crezcamos,2022-12-30,Natural,Masculino,Microempresa,Microcredito,Garantia idónea o no idónea,Microcrédito entre 25 y 120 SMMLV,Hasta 1 año,32.0000,0.0000,40000000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-12,40000000.0000
15034,BC-ESTABLECIMIENTO BANCARIO,Itaú,2022-07-22,Jurídica,No aplica,Gran empresa,Comercial ordinario,Garantia idónea o no idónea,Empresarial en pesos,Más de 30 días y hasta 1 año,12.9300,0.0000,16550000000.0000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07,4137500000.0000
60076,BC-ESTABLECIMIENTO BANCARIO,AV Villas,2022-10-14,Natural,Femenino,No aplica,Consumo,Sin garantia,Tarjeta de crédito para ingresos superiores a 2 SMMLV,Consumos entre 2 y 6 meses,35.7800,0.0000,2690984624.6200,9025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-10,298170.0415


In [11]:
tasascap[tasascap["anomes"] == "2022-08"]["fecha_corte"].value_counts()

fecha_corte
2022-08-26    4077
2022-08-05    3956
2022-08-12    3906
2022-08-19    3833
Name: count, dtype: int64

In [ ]:
utils.vc(tasascap["producto"])

In [ ]:
tasascap.isnull().mean()

In [9]:
print(np.percentile(tasascap["tasa"], 2.25))
print(np.percentile(tasascap["tasa"], 5))
print(np.percentile(tasascap["tasa"], 95))
print(np.percentile(tasascap["tasa"], 97.5))

4.91
10.7
41.125
44.25


In [ ]:
print(tasascap[tasascap["tasa"] <= 10].shape)
print(tasascap[tasascap["tasa"] >= 40].shape)

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("producto").agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
ax = ax.sort_values("num_obl", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("banco").agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
ax = ax.sort_values("num_obl", ascending=False)#.head(30)
ax

In [ ]:
tasascap["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascap
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascap
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_obl":"sum", "tasa":"mean"}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_obl"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascap = ax
tasascap

## How to group?

## BANREP

In [12]:
banrep1 = pd.read_excel(path_data+"BANREP/tasas de interes y sector - tasas interes.xlsx", decimal = ",")
banrep1 = banrep1.iloc[1:]
banrep1.columns = [x.lower().replace("í","i").replace("ó","o").replace("é","e").replace(" ", "_") for x in banrep1.columns]
banrep1 = banrep1.replace({'-':np.nan})
banrep1["fecha"] = pd.to_datetime(banrep1["fecha"], format="%d/%m/%Y", errors='coerce')
banrep1["periodo"] = banrep1["fecha"].dt.strftime("%Y-%m")

print(banrep1["fecha"].min(), "|", banrep1["fecha"].max())
print(banrep1.shape)
banrep1

2022-01-03 00:00:00 | 2026-10-16 00:00:00
(1196, 22)


,fecha,"credito_de_consumo,_tasa_de_interes,_semanal(dato_semanal)","microcreditos_(diferentes_de_leasing),_tasa_de_interes,_semanal(dato_semanal)","creditos_comerciales_(ordinario),_tasa_de_interes,_semanal(dato_semanal)","creditos_comerciales_(preferencial_o_corporativo),_tasa_de_interes,_semanal(dato_semanal)","indicador_bancario_de_referencia_(ibr)_overnight,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_overnight,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_1_mes,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_1_mes,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_3_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_3_meses,_efectiva(promedio_semanal)",operaciones_interbancarias_-_monto(dato_diario),"indicador_bancario_de_referencia_(ibr)_a_6_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_6_meses,_efectiva(promedio_semanal)","indicador_bancario_de_referencia_(ibr)_a_12_meses,_efectiva(dato_diario)","indicador_bancario_de_referencia_(ibr)_a_12_meses,_efectiva(promedio_semanal)","credito_productivo_(total),_semanal(dato_semanal)","credito_productivo_de_mayor_monto(total),_semanal(dato_semanal)","credito_popular_productivo_(total),_mensual(promedio_mensual)","captaciones_a_traves_de_cdt_por_red_de_oficinas,_diaria(dato_diario)","captaciones_a_traves_de_cdt_por_tesoreria,_diaria(dato_diario)",periodo
1,2022-01-03,NaN,NaN,NaN,NaN,2.998,NaN,3.087,NaN,3.586,NaN,"915.900,00",4.274,NaN,NaN,NaN,NaN,NaN,NaN,3.49,4.34,2022-01
2,2022-01-04,NaN,NaN,NaN,NaN,2.981,NaN,3.140,NaN,3.644,NaN,"715.500,00",4.335,NaN,NaN,NaN,NaN,NaN,NaN,3.73,5.23,2022-01
3,2022-01-05,NaN,NaN,NaN,NaN,2.995,NaN,3.168,NaN,3.660,NaN,"745.600,00",4.377,NaN,NaN,NaN,NaN,NaN,NaN,3.75,4.02,2022-01
4,2022-01-06,NaN,NaN,NaN,NaN,2.998,NaN,3.218,NaN,3.711,NaN,"835.300,00",4.512,NaN,NaN,NaN,NaN,NaN,NaN,3.53,5.13,2022-01
5,2022-01-07,16.87,39.24,9.17,6.20,3.000,2.994,3.286,3.180,3.773,3.675,"979.300,00",4.611,4.422,NaN,NaN,NaN,NaN,NaN,3.73,4.52,2022-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,2026-10-15,NaN,NaN,NaN,NaN,12.260,NaN,12.332,NaN,12.388,NaN,"1.207.853,08",12.481,NaN,12.599,NaN,NaN,NaN,NaN,10.70,12.92,2026-10
1193,2026-10-16,NaN,NaN,NaN,NaN,12.257,12.257,12.295,12.295,12.382,12.382,NaN,12.468,12.468,12.563,12.563,NaN,NaN,NaN,NaN,NaN,2026-10
1194,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1195,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
